In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_real_estate_research_qca(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "RealEstateResearchId string, SourceSystem string, PortfolioKey string, PortfolioName string,"
        #     "IngestionTimestamp timestamp, BatchId string, RealEstateResearchKey int,"
        #     "PartyKey int, RealEstateResearchStatusCode string, RealEstateResearchStatus string,"
        #     "RealEstateResearchDate timestamp, CreatedDate string, CompanyCode int, Company int,"
        #     "Comments string, OutcomeCode int, Outcome string, CancellationUser string,"
        #     "RecallDate timestamp,"
        #     "RealEstateResearchTypeCode string,"
        #     "RealEstateResearchType string,"
        #     "PacketCode int,"
        #     "Packet string,"
        #     "StreamCycleCode int,"
        #     "StreamCycle int,"
        #     "IsCompletedFlag boolean,"
        #     "LandRegistryCode string,"
        #     "LandRegistry string,"
        #     "LandOfficeCode string,"
        #     "LandOffice string,"
        #     "EAPSourceKey string,"
        #     "CancelledFlag int,"
        #     "CancelledDate timestamp,"
        #     "CancelledReasonCode string,"
        #     "CancelledReason string,"
        #     "TotalDays int,"
        #     "RealEstateResearchGroupStatusCode string,"
        #     "RealEstateResearchGroupStatus string,"
        #     "RealEstateResearchPerson int,"
        #     "RealEstateResearchPersonAddress string,"
        #     "RealEstateResearchPersonCounty string,"
        #     "RealEstateResearchRequestDate timestamp,"
        #     "RealEstateResearchAssignmentKey int,"
        #     "RealEstateResearchAssignmentOperator int,"
        #     "RealEstateResearchPriority boolean,"
        #     "RealEstateResearchRecallReason string,"
        #     "RealEstateResearchArea string,"
        #     "RealEstateResearchFirstDebtorCDI string,"
        #     "RealEstateResearchAssignmentPool string,"
        #     "RealEstateResearchRequestorComments string,"
        #     "RealEstateResearchPriorityReason string,"
        #     "RealEstateResearchAssignmentDate timestamp,"
        #     "RealEstateResearchRequestSystem string,"
        #     "RealEstateResearchReviewedDate timestamp,"
        #     "RealEstateResearchValidToDate timestamp,"
        #     "RealEstateResearcSubmittedBy int,"
        #     "RealEstateResearchPersonTaxRegistrationNumber string,"
        #     "HasRealEstateResearchProcessesed boolean,"
        #     "RealEstateResearchGroupCode int,"
        #     "RealEstateResearchRequestCode int,"
        #     "HasPerformedNewRealEstateResearchFlag int,"
        #     "RealEstateResearchExposureExpenses boolean,"
        #     "RealEstateResearchLastUpdatedDate timestamp,"
        #     "RealEstateKey int,"
        #     "RealEstateResearchExpiredDate timestamp,"
        #     "RealEstateResearchRiskStatus string,"
        #     "RealEstateResearchReferencekey string,"
        #     "RealEstateResearchRemediationStatus string,"
        #     "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.RealEstateResearch"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "qca",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "portfolio_id_qca",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== DIM_PROPERTY_ENQUIRY_LAND_OFFICE ==================

        dim_property_enquiry_land_office_extra_filters = None

        if dim_property_enquiry_land_office_extra_filters is not None:
            dim_property_enquiry_land_office_extra_filters.append(date_filter)
            dim_property_enquiry_land_office_final_filters = " AND ".join(
                dim_property_enquiry_land_office_extra_filters
            )
        else:
            dim_property_enquiry_land_office_final_filters = date_filter

        dim_property_enquiry_land_office_selected_fields = [
            "PENLO_ID",
            "PENLO_COMMENTS",
        ]

        dim_property_enquiry_land_office = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_property_enquiry_land_office",
            dim_property_enquiry_land_office_final_filters,
            dim_property_enquiry_land_office_selected_fields,
        )

        # ================== DIM_PACKET ==================

        dim_packet_extra_filters = None

        if dim_packet_extra_filters is not None:
            dim_packet_extra_filters.append(date_filter)
            dim_packet_final_filters = " AND ".join(dim_packet_extra_filters)
        else:
            dim_packet_final_filters = date_filter

        dim_packet_selected_fields = [
            "PACK_ID",
            "PACK_NAME",
        ]

        dim_packet = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_packet",
            dim_packet_final_filters,
            dim_packet_selected_fields,
        )

        # ================== DIM_PROPERTY_ENQUIRY_LAND_REGISTRY ==================

        dim_property_enquiry_land_registry_extra_filters = None

        if dim_property_enquiry_land_registry_extra_filters is not None:
            dim_property_enquiry_land_registry_extra_filters.append(date_filter)
            dim_property_enquiry_land_registry_final_filters = " AND ".join(
                dim_property_enquiry_land_registry_extra_filters
            )
        else:
            dim_property_enquiry_land_registry_final_filters = date_filter

        dim_property_enquiry_land_registry_selected_fields = [
            "PENLR_COMMENTS",
            "PENLR_ID",
        ]

        dim_property_enquiry_land_registry = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_property_enquiry_land_registry",
            dim_property_enquiry_land_registry_final_filters,
            dim_property_enquiry_land_registry_selected_fields,
        )

        dim_property_enquiry_land_registry = dim_property_enquiry_land_registry.select(
            "PENLR_COMMENTS", "PENLR_ID"
        ).distinct()

        # ================== DIM_CUSTOMER_STATIC ==================

        dim_customer_static_extra_filters = None

        if dim_customer_static_extra_filters is not None:
            dim_customer_static_extra_filters.append(date_filter)
            dim_customer_static_final_filters = " AND ".join(
                dim_customer_static_extra_filters
            )
        else:
            dim_customer_static_final_filters = date_filter

        dim_customer_static_selected_fields = [
            "CUST_ID",
            "CUST_CIF",
        ]

        dim_customer_static = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_customer_static",
            dim_customer_static_final_filters,
            dim_customer_static_selected_fields,
        )

        # ============================== FS_CASES_SNAP ====================

        FS_CASES_SNAP_extra_filters = None

        if FS_CASES_SNAP_extra_filters is not None:
            FS_CASES_SNAP_extra_filters.append(date_filter)
            FS_CASES_SNAP_final_filters = " AND ".join(FS_CASES_SNAP_extra_filters)
        else:
            FS_CASES_SNAP_final_filters = date_filter

        FS_CASES_SNAP_selected_fields = ["CUST_CODE"]

        FS_CASES_SNAP = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_FS_CASES_SNAP",
            FS_CASES_SNAP_final_filters,
            FS_CASES_SNAP_selected_fields,
        )

        FS_CASES_SNAP = FS_CASES_SNAP.select(FS_CASES_SNAP.CUST_CODE).distinct()

        # ================== FS_REAL_ESTATE_INVESTIGATIONS ==================

        FS_REAL_ESTATE_INVESTIGATIONS_extra_filters = None

        if FS_REAL_ESTATE_INVESTIGATIONS_extra_filters is not None:
            FS_REAL_ESTATE_INVESTIGATIONS_extra_filters.append(date_filter)
            FS_REAL_ESTATE_INVESTIGATIONS_final_filters = " AND ".join(
                FS_REAL_ESTATE_INVESTIGATIONS_extra_filters
            )
        else:
            FS_REAL_ESTATE_INVESTIGATIONS_final_filters = date_filter

        FS_REAL_ESTATE_INVESTIGATIONS_selected_fields = [
            "REIN_CODE",
            "PD_DESC",
            "CUST_CODE",
            "REIN_EXTC_ID",
            "EXTC_ID",
            "REIN_PROJECT_ID",
            "REIN_CANCELATION_USER",
            "PACC_DATEBACK",
            "PENO_ID",
            "PENLO_ID",
            "PENLR_ID",
            "SC_ID",
            "PACK_ID",
            "REIN_WORK_TYPE_INTCODE",
            "PACC_IS_COMPLETED_FLAG",
            "REIN_SOURCE",
            "INDX_CANCELLED_ID",
            "PENCR_ID",
            "businessdate",
        ]

        FS_REAL_ESTATE_INVESTIGATIONS = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_FS_REAL_ESTATE_INVESTIGATIONS",
            FS_REAL_ESTATE_INVESTIGATIONS_final_filters,
            FS_REAL_ESTATE_INVESTIGATIONS_selected_fields,
        )

        # ================== DIM_PROPERTY_ENQUIRY_STATUS ==================

        DIM_PROPERTY_ENQUIRY_STATUS_extra_filters = None

        if DIM_PROPERTY_ENQUIRY_STATUS_extra_filters is not None:
            DIM_PROPERTY_ENQUIRY_STATUS_extra_filters.append(date_filter)
            DIM_PROPERTY_ENQUIRY_STATUS_final_filters = " AND ".join(
                DIM_PROPERTY_ENQUIRY_STATUS_extra_filters
            )
        else:
            DIM_PROPERTY_ENQUIRY_STATUS_final_filters = date_filter

        DIM_PROPERTY_ENQUIRY_STATUS_selected_fields = [
            "PES_DESC",
            "PES_ID",
        ]

        DIM_PROPERTY_ENQUIRY_STATUS = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qca_DIM_PROPERTY_ENQUIRY_STATUS",
            DIM_PROPERTY_ENQUIRY_STATUS_final_filters,
            DIM_PROPERTY_ENQUIRY_STATUS_selected_fields,
        )

        # ================== FS_PROPERTY_ENQUIRY_SNAP ==================

        FS_PROPERTY_ENQUIRY_SNAP_extra_filters = None

        if FS_PROPERTY_ENQUIRY_SNAP_extra_filters is not None:
            FS_PROPERTY_ENQUIRY_SNAP_extra_filters.append(date_filter)
            FS_PROPERTY_ENQUIRY_SNAP_final_filters = " AND ".join(
                FS_PROPERTY_ENQUIRY_SNAP_extra_filters
            )
        else:
            FS_PROPERTY_ENQUIRY_SNAP_final_filters = date_filter

        FS_PROPERTY_ENQUIRY_SNAP_selected_fields = [
            "WKFV_ID",
            "SC_ID",
            "CUST_CODE",
            "PES_ID",
            "SC_DAYS_ID",
        ]

        FS_PROPERTY_ENQUIRY_SNAP = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qca_FS_PROPERTY_ENQUIRY_SNAP",
            FS_PROPERTY_ENQUIRY_SNAP_final_filters,
            FS_PROPERTY_ENQUIRY_SNAP_selected_fields,
        )

        # ================== DIM_PROPERTY_ENQUIRY_OUTCOME ==================

        DIM_PROPERTY_ENQUIRY_OUTCOME_extra_filters = None

        if DIM_PROPERTY_ENQUIRY_OUTCOME_extra_filters is not None:
            DIM_PROPERTY_ENQUIRY_OUTCOME_extra_filters.append(date_filter)
            DIM_PROPERTY_ENQUIRY_OUTCOME_final_filters = " AND ".join(
                DIM_PROPERTY_ENQUIRY_OUTCOME_extra_filters
            )
        else:
            DIM_PROPERTY_ENQUIRY_OUTCOME_final_filters = date_filter

        DIM_PROPERTY_ENQUIRY_OUTCOME_selected_fields = ["PENO_ID", "PENO_DESC"]

        DIM_PROPERTY_ENQUIRY_OUTCOME = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qca_DIM_PROPERTY_ENQUIRY_OUTCOME",
            DIM_PROPERTY_ENQUIRY_OUTCOME_final_filters,
            DIM_PROPERTY_ENQUIRY_OUTCOME_selected_fields,
        )

        # ================== DIM_PROPERTY_ENQUIRY_CANCELLATION_REASON ==================

        DIM_PROPERTY_ENQUIRY_CANCELLATION_REASON_extra_filters = None

        if DIM_PROPERTY_ENQUIRY_CANCELLATION_REASON_extra_filters is not None:
            DIM_PROPERTY_ENQUIRY_CANCELLATION_REASON_extra_filters.append(date_filter)
            DIM_PROPERTY_ENQUIRY_CANCELLATION_REASON_final_filters = " AND ".join(
                DIM_PROPERTY_ENQUIRY_CANCELLATION_REASON_extra_filters
            )
        else:
            DIM_PROPERTY_ENQUIRY_CANCELLATION_REASON_final_filters = date_filter

        DIM_PROPERTY_ENQUIRY_CANCELLATION_REASON_selected_fields = [
            "PENCR_ID",
            "CR_DATE",
            "PENCR_DESC",
        ]

        DIM_PROPERTY_ENQUIRY_CANCELLATION_REASON = (
            load_landed_delta_table(  # noqa: F841
                spark,
                catalog_landed,  # type: ignore
                "qca_DIM_PROPERTY_ENQUIRY_CANCELLATION_REASON",
                DIM_PROPERTY_ENQUIRY_CANCELLATION_REASON_final_filters,
                DIM_PROPERTY_ENQUIRY_CANCELLATION_REASON_selected_fields,
            )
        )

        # ================== DIM_EXTERNAL_COMPANY ==================

        DIM_EXTERNAL_COMPANY_extra_filters = None

        if DIM_EXTERNAL_COMPANY_extra_filters is not None:
            DIM_EXTERNAL_COMPANY_extra_filters.append(date_filter)
            DIM_EXTERNAL_COMPANY_final_filters = " AND ".join(
                DIM_EXTERNAL_COMPANY_extra_filters
            )
        else:
            DIM_EXTERNAL_COMPANY_final_filters = date_filter

        DIM_EXTERNAL_COMPANY_selected_fields = [
            "EXTC_ID",
            "EXTC_NAME",
        ]

        DIM_EXTERNAL_COMPANY = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qca_DIM_EXTERNAL_COMPANY",
            DIM_EXTERNAL_COMPANY_final_filters,
            DIM_EXTERNAL_COMPANY_selected_fields,
        )

        # Join conditions
        FS_REAL_ESTATE_INVESTIGATIONS_DIM_PROPERTY_ENQUIRY_CANCELLATION_REASON_join_condition = (
            FS_REAL_ESTATE_INVESTIGATIONS["PENCR_ID"]
            == DIM_PROPERTY_ENQUIRY_CANCELLATION_REASON["PENCR_ID"]
        )

        FS_REAL_ESTATE_INVESTIGATIONS_DIM_EXTERNAL_COMPANY_join_condition = (
            DIM_EXTERNAL_COMPANY["EXTC_ID"]
            == FS_REAL_ESTATE_INVESTIGATIONS["REIN_EXTC_ID"]
        )

        FS_REAL_ESTATE_INVESTIGATIONS_DIM_PROPERTY_ENQUIRY_OUTCOME_join_condition = (
            FS_REAL_ESTATE_INVESTIGATIONS["PENO_ID"]
            == DIM_PROPERTY_ENQUIRY_OUTCOME["PENO_ID"]
        )

        FS_REAL_ESTATE_INVESTIGATIONS_FS_PROPERTY_ENQUIRY_SNAP_join_condition = (
            FS_REAL_ESTATE_INVESTIGATIONS["SC_ID"] == FS_PROPERTY_ENQUIRY_SNAP["SC_ID"]
        ) & (
            FS_REAL_ESTATE_INVESTIGATIONS["CUST_CODE"]
            == FS_PROPERTY_ENQUIRY_SNAP["CUST_CODE"]
        )

        FS_PROPERTY_ENQUIRY_SNAP_DIM_PROPERTY_ENQUIRY_STATUS_join_condition = (
            FS_PROPERTY_ENQUIRY_SNAP["PES_ID"] == DIM_PROPERTY_ENQUIRY_STATUS["PES_ID"]
        )

        DIM_PROPERTY_ENQUIRY_LAND_OFFICE_FS_REAL_ESTATE_INVESTIGATIONS_join_condition = (
            dim_property_enquiry_land_office["PENLO_ID"]
            == FS_REAL_ESTATE_INVESTIGATIONS["PENLO_ID"]
        )

        DIM_PACKET_FS_REAL_ESTATE_INVESTIGATIONS_join_condition = (
            dim_packet["PACK_ID"] == FS_REAL_ESTATE_INVESTIGATIONS["PACK_ID"]
        )

        DIM_PROPERTY_ENQUIRY_LAND_REGISTRY_FS_REAL_ESTATE_INVESTIGATIONS_join_condition = (
            dim_property_enquiry_land_registry["PENLR_ID"]
            == FS_REAL_ESTATE_INVESTIGATIONS["PENLR_ID"]
        )

        FS_CASES_SNAP_FS_REAL_ESTATE_INVESTIGATIONS_join_condition = (
            FS_CASES_SNAP["CUST_CODE"] == FS_REAL_ESTATE_INVESTIGATIONS["CUST_CODE"]
        )

        DIM_CUSTOMER_STATIC_FS_CASES_SNAP_join_condition = (
            dim_customer_static["CUST_ID"] == FS_CASES_SNAP["CUST_CODE"]
        )

        portfolio_DIM_CUSTOMER_STATIC_join_condition = portfolio["Prefix"] == substring(
            dim_customer_static["CUST_CIF"], 1, 3
        )

        # Joins

        target_table_qca = (
            FS_REAL_ESTATE_INVESTIGATIONS.join(
                DIM_PROPERTY_ENQUIRY_OUTCOME,
                FS_REAL_ESTATE_INVESTIGATIONS_DIM_PROPERTY_ENQUIRY_OUTCOME_join_condition,
                "inner",
            )
            .join(
                FS_CASES_SNAP,
                FS_CASES_SNAP_FS_REAL_ESTATE_INVESTIGATIONS_join_condition,
                "inner",
            )
            .join(
                dim_customer_static,
                DIM_CUSTOMER_STATIC_FS_CASES_SNAP_join_condition,
                "inner",
            )
            .join(
                DIM_PROPERTY_ENQUIRY_CANCELLATION_REASON,
                FS_REAL_ESTATE_INVESTIGATIONS_DIM_PROPERTY_ENQUIRY_CANCELLATION_REASON_join_condition,
                "left",
            )
            .join(
                DIM_EXTERNAL_COMPANY,
                FS_REAL_ESTATE_INVESTIGATIONS_DIM_EXTERNAL_COMPANY_join_condition,
                "left",
            )
            .join(
                FS_PROPERTY_ENQUIRY_SNAP,
                FS_REAL_ESTATE_INVESTIGATIONS_FS_PROPERTY_ENQUIRY_SNAP_join_condition,
                "left",
            )
            .join(
                DIM_PROPERTY_ENQUIRY_STATUS,
                FS_PROPERTY_ENQUIRY_SNAP_DIM_PROPERTY_ENQUIRY_STATUS_join_condition,
                "left",
            )
            .join(
                dim_property_enquiry_land_office,
                DIM_PROPERTY_ENQUIRY_LAND_OFFICE_FS_REAL_ESTATE_INVESTIGATIONS_join_condition,
                "left",
            )
            .join(
                dim_packet,
                DIM_PACKET_FS_REAL_ESTATE_INVESTIGATIONS_join_condition,
                "left",
            )
            .join(
                dim_property_enquiry_land_registry,
                DIM_PROPERTY_ENQUIRY_LAND_REGISTRY_FS_REAL_ESTATE_INVESTIGATIONS_join_condition,
                "left",
            )
            .join(
                portfolio,
                portfolio_DIM_CUSTOMER_STATIC_join_condition,
                "LEFT",
            )
        )

        # HashKeys
        target_table_qca_RealEstateResearchId_primary_key_fields = [
            FS_REAL_ESTATE_INVESTIGATIONS["REIN_CODE"],
            lit("qca"),
        ]

        # Target table (hash keys and calculations)
        target_table_qca = target_table_qca.withColumn(
            "RealEstateResearchId",
            sha2(
                concat_ws(
                    ";", *target_table_qca_RealEstateResearchId_primary_key_fields
                ),
                256,
            ),
        )
        # ================== Selection of Final Columns ==================

        result_df_qca = target_table_qca.select(
            target_table_qca.RealEstateResearchId.alias("RealEstateResearchId"),
            lit("qca").cast("string").alias("SourceSystem"),
            coalesce(portfolio.portfolio_id_qca, lit(2))
            .cast("string")
            .alias("PortfolioKey"),
            coalesce(portfolio.PortfolioName, lit("Core Retail"))
            .cast("string")
            .alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("string").alias("BatchId"),
            FS_REAL_ESTATE_INVESTIGATIONS.REIN_CODE.cast("int").alias(
                "RealEstateResearchKey"
            ),
            FS_REAL_ESTATE_INVESTIGATIONS.CUST_CODE.cast("int").alias("PartyKey"),
            FS_PROPERTY_ENQUIRY_SNAP.PES_ID.cast("string").alias(
                "RealEstateResearchStatusCode"
            ),
            DIM_PROPERTY_ENQUIRY_STATUS.PES_DESC.cast("string").alias(
                "RealEstateResearchStatus"
            ),
            FS_REAL_ESTATE_INVESTIGATIONS.PD_DESC.cast("timestamp").alias(
                "RealEstateResearchDate"
            ),
            FS_PROPERTY_ENQUIRY_SNAP.WKFV_ID.cast("string").alias("CreatedDate"),
            FS_REAL_ESTATE_INVESTIGATIONS.EXTC_ID.cast("int").alias("CompanyCode"),
            DIM_EXTERNAL_COMPANY.EXTC_NAME.cast("int").alias("Company"),
            FS_REAL_ESTATE_INVESTIGATIONS.REIN_PROJECT_ID.cast("string").alias(
                "Comments"
            ),
            DIM_PROPERTY_ENQUIRY_OUTCOME.PENO_ID.cast("int").alias("OutcomeCode"),
            DIM_PROPERTY_ENQUIRY_OUTCOME.PENO_DESC.cast("string").alias("Outcome"),
            FS_REAL_ESTATE_INVESTIGATIONS.REIN_CANCELATION_USER.cast("string").alias(
                "CancellationUser"
            ),
            FS_REAL_ESTATE_INVESTIGATIONS.PACC_DATEBACK.cast("timestamp").alias(
                "RecallDate"
            ),
            FS_REAL_ESTATE_INVESTIGATIONS.REIN_WORK_TYPE_INTCODE.cast("string").alias(
                "RealEstateResearchTypeCode"
            ),
            FS_REAL_ESTATE_INVESTIGATIONS.REIN_WORK_TYPE_INTCODE.cast("string").alias(
                "RealEstateResearchType"
            ),
            FS_REAL_ESTATE_INVESTIGATIONS.PACK_ID.cast("int").alias("PacketCode"),
            dim_packet.PACK_NAME.cast("string").alias("Packet"),
            FS_REAL_ESTATE_INVESTIGATIONS.SC_ID.cast("int").alias("StreamCycleCode"),
            FS_REAL_ESTATE_INVESTIGATIONS.SC_ID.cast("int").alias("StreamCycle"),
            FS_REAL_ESTATE_INVESTIGATIONS.PACC_IS_COMPLETED_FLAG.cast("boolean").alias(
                "IsCompletedFlag"
            ),
            FS_REAL_ESTATE_INVESTIGATIONS.PENLR_ID.cast("string").alias(
                "LandRegistryCode"
            ),
            dim_property_enquiry_land_registry.PENLR_COMMENTS.cast("string").alias(
                "LandRegistry"
            ),
            FS_REAL_ESTATE_INVESTIGATIONS.PENLO_ID.cast("string").alias(
                "LandOfficeCode"
            ),
            dim_property_enquiry_land_office.PENLO_COMMENTS.cast("string").alias(
                "LandOffice"
            ),
            FS_REAL_ESTATE_INVESTIGATIONS.REIN_SOURCE.cast("string").alias(
                "EAPSourceKey"
            ),
            FS_REAL_ESTATE_INVESTIGATIONS.INDX_CANCELLED_ID.cast("int").alias(
                "CancelledFlag"
            ),
            DIM_PROPERTY_ENQUIRY_CANCELLATION_REASON.CR_DATE.cast("timestamp").alias(
                "CancelledDate"
            ),
            FS_REAL_ESTATE_INVESTIGATIONS.PENCR_ID.cast("string").alias(
                "CancelledReasonCode"
            ),
            DIM_PROPERTY_ENQUIRY_CANCELLATION_REASON.PENCR_DESC.cast("string").alias(
                "CancelledReason"
            ),
            FS_PROPERTY_ENQUIRY_SNAP.SC_DAYS_ID.cast("int").alias("TotalDays"),
            lit(None).alias("RealEstateResearchGroupStatusCode"),
            lit(None).alias("RealEstateResearchGroupStatus"),
            lit(None).alias("RealEstateResearchPerson"),
            lit(None).alias("RealEstateResearchPersonAddress"),
            lit(None).alias("RealEstateResearchPersonCounty"),
            lit(None).alias("RealEstateResearchRequestDate"),
            lit(None).alias("RealEstateResearchAssignmentKey"),
            lit(None).alias("RealEstateResearchAssignmentOperator"),
            lit(None).alias("RealEstateResearchPriority"),
            lit(None).alias("RealEstateResearchRecallReason"),
            lit(None).alias("RealEstateResearchArea"),
            lit(None).alias("RealEstateResearchFirstDebtorCDI"),
            lit(None).alias("RealEstateResearchAssignmentPool"),
            lit(None).alias("RealEstateResearchRequestorComments"),
            lit(None).alias("RealEstateResearchPriorityReason"),
            lit(None).alias("RealEstateResearchAssignmentDate"),
            lit(None).alias("RealEstateResearchRequestSystem"),
            lit(None).alias("RealEstateResearchReviewedDate"),
            lit(None).alias("RealEstateResearchValidToDate"),
            lit(None).alias("RealEstateResearcSubmittedBy"),
            lit(None).alias("RealEstateResearchPersonTaxRegistrationNumber"),
            lit(None).alias("HasRealEstateResearchProcessesed"),
            lit(None).alias("RealEstateResearchGroupCode"),
            lit(None).alias("RealEstateResearchRequestCode"),
            lit(None).alias("HasPerformedNewRealEstateResearchFlag"),
            lit(None).alias("RealEstateResearchExposureExpenses"),
            lit(None).alias("RealEstateResearchLastUpdatedDate"),
            lit(None).alias("RealEstateKey"),
            lit(None).alias("RealEstateResearchExpiredDate"),
            lit(None).alias("RealEstateResearchRiskStatus"),
            lit(None).alias("RealEstateResearchReferencekey"),
            lit(None).alias("RealEstateResearchRemediationStatus"),
            FS_REAL_ESTATE_INVESTIGATIONS.businessdate.cast("date").alias(
                "BusinessDate"
            ),
        ).distinct()


        # ================== Append ==================
        
        result_df_qca.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_real_estate_research_qca(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)